In [1]:
LAST_NAME = "Wales"
STUDENT_ID = "TUPM-26-7925"
FAVORITE_ARTIST = "IVOS"

id_digits = [int(c) for c in STUDENT_ID if c.isdigit()]
letters = [c for c in FAVORITE_ARTIST.upper() if c.isalpha()]

SEED_NUM = id_digits[-1]    # CHECK this against the SEED_NUM rule in your Lab 4 handout

print("SEED_NUM        :", SEED_NUM)
print("LAST_NAME       :", LAST_NAME)
print("FAVORITE_ARTIST :", FAVORITE_ARTIST)

SEED_NUM        : 5
LAST_NAME       : Wales
FAVORITE_ARTIST : IVOS


In [2]:
students = [f"{LAST_NAME}-S{i}" for i in range(1, 6)]
subjects = ["PROGRAMMING", "MATHEMATICS", "ELECTRONICS", "PHYSICS", "ENGLISH"]

counter = 0
def next_score():
    """Score from SEED_NUM and the next character position of FAVORITE_ARTIST (50-100)."""
    global counter
    pos = counter % len(letters) + 1
    letter_value = ord(letters[pos - 1]) - ord("A") + 1
    counter += 1
    raw = letter_value * pos + SEED_NUM
    return 50 + (raw * 7) % 51

def category(score):
    if score >= 90: return "Excellent"
    if score >= 80: return "Good"
    if score >= 70: return "Fair"
    return "Needs Improvement"

records = []

# each student gets three subjects
for i, student in enumerate(students):
    for j in range(3):
        records.append({"student": student, "subject": subjects[(i + j) % 5], "score": next_score()})

# one repeated student-subject record (exact copy of the first record)
records.append(dict(records[0]))

# one incomplete record (missing score)
records.append({"student": students[4], "subject": subjects[(4 + 3) % 5], "score": None})

print(f"{'#':<4}{'Student':<10}{'Subject':<14}{'Score':>8}")
for n, r in enumerate(records, 1):
    shown = r["score"] if r["score"] is not None else "MISSING"
    print(f"{n:<4}{r['student']:<10}{r['subject']:<14}{shown:>8}")

#   Student   Subject          Score
1   Wales-S1  PROGRAMMING         97
2   Wales-S1  MATHEMATICS         87
3   Wales-S1  ELECTRONICS         94
4   Wales-S2  MATHEMATICS         56
5   Wales-S2  ELECTRONICS         97
6   Wales-S2  PHYSICS             87
7   Wales-S3  ELECTRONICS         94
8   Wales-S3  PHYSICS             56
9   Wales-S3  ENGLISH             97
10  Wales-S4  PHYSICS             87
11  Wales-S4  ENGLISH             94
12  Wales-S4  PROGRAMMING         56
13  Wales-S5  ENGLISH             97
14  Wales-S5  PROGRAMMING         87
15  Wales-S5  MATHEMATICS         94
16  Wales-S1  PROGRAMMING         97
17  Wales-S5  ELECTRONICS    MISSING


In [3]:
from collections import Counter

key_counts = Counter((r["student"], r["subject"]) for r in records)
repeated = [k for k, c in key_counts.items() if c > 1]
incomplete = [r for r in records if r["score"] is None]

print("Total records     :", len(records))
print("Repeated records  :", repeated)
print("Incomplete records:", [(r["student"], r["subject"]) for r in incomplete])

Total records     : 17
Repeated records  : [('Wales-S1', 'PROGRAMMING')]
Incomplete records: [('Wales-S5', 'ELECTRONICS')]


In [ ]:
# Keep only valid records, and count each student-subject pair once.
seen, valid = set(), []
for r in records:
    key = (r["student"], r["subject"])
    if r["score"] is None or key in seen:
        continue
    seen.add(key)
    valid.append(r)

by_student, by_subject = {}, {}
for r in valid:
    by_student.setdefault(r["student"], {})[r["subject"]] = r["score"]
    by_subject.setdefault(r["subject"], {})[r["student"]] = r["score"]

print("Valid, unique records:", len(valid))
print("\nBy student:")
for st, subs in by_student.items():
    print(f"  {st}: {subs}")
print("\nBy subject:")
for sj, studs in by_subject.items():
    print(f"  {sj}: {studs}")

Valid, unique records: 15

By student:
  Wales-S1: {'PROGRAMMING': 97, 'MATHEMATICS': 87, 'ELECTRONICS': 94}
  Wales-S2: {'MATHEMATICS': 56, 'ELECTRONICS': 97, 'PHYSICS': 87}
  Wales-S3: {'ELECTRONICS': 94, 'PHYSICS': 56, 'ENGLISH': 97}
  Wales-S4: {'PHYSICS': 87, 'ENGLISH': 94, 'PROGRAMMING': 56}
  Wales-S5: {'ENGLISH': 97, 'PROGRAMMING': 87, 'MATHEMATICS': 94}

By subject:
  PROGRAMMING: {'Wales-S1': 97, 'Wales-S4': 56, 'Wales-S5': 87}
  MATHEMATICS: {'Wales-S1': 87, 'Wales-S2': 56, 'Wales-S5': 94}
  ELECTRONICS: {'Wales-S1': 94, 'Wales-S2': 97, 'Wales-S3': 94}
  PHYSICS: {'Wales-S2': 87, 'Wales-S3': 56, 'Wales-S4': 87}
  ENGLISH: {'Wales-S3': 97, 'Wales-S4': 94, 'Wales-S5': 97}


In [5]:
for r in valid:
    r["category"] = category(r["score"])

print(f"{'Student':<10}{'Subject':<14}{'Score':>6}  Category")
for r in valid:
    print(f"{r['student']:<10}{r['subject']:<14}{r['score']:>6}  {r['category']}")

print("\nCategory counts:", dict(Counter(r["category"] for r in valid)))

Student   Subject        Score  Category
Wales-S1  PROGRAMMING       97  Excellent
Wales-S1  MATHEMATICS       87  Good
Wales-S1  ELECTRONICS       94  Excellent
Wales-S2  MATHEMATICS       56  Needs Improvement
Wales-S2  ELECTRONICS       97  Excellent
Wales-S2  PHYSICS           87  Good
Wales-S3  ELECTRONICS       94  Excellent
Wales-S3  PHYSICS           56  Needs Improvement
Wales-S3  ENGLISH           97  Excellent
Wales-S4  PHYSICS           87  Good
Wales-S4  ENGLISH           94  Excellent
Wales-S4  PROGRAMMING       56  Needs Improvement
Wales-S5  ENGLISH           97  Excellent
Wales-S5  PROGRAMMING       87  Good
Wales-S5  MATHEMATICS       94  Excellent

Category counts: {'Excellent': 8, 'Good': 4, 'Needs Improvement': 3}


In [6]:
def summarize(groups):
    out = {}
    for name, items in groups.items():
        sc = list(items.values())
        out[name] = {"count": len(sc), "avg": sum(sc) / len(sc), "high": max(sc), "low": min(sc)}
    return out

student_summary = summarize(by_student)
subject_summary = summarize(by_subject)

print(f"{'Student':<10}{'Count':>6}{'Avg':>8}{'High':>6}{'Low':>6}")
for k, s in student_summary.items():
    print(f"{k:<10}{s['count']:>6}{s['avg']:>8.2f}{s['high']:>6}{s['low']:>6}")

print(f"\n{'Subject':<14}{'Count':>6}{'Avg':>8}{'High':>6}{'Low':>6}")
for k, s in subject_summary.items():
    print(f"{k:<14}{s['count']:>6}{s['avg']:>8.2f}{s['high']:>6}{s['low']:>6}")

Student    Count     Avg  High   Low
Wales-S1       3   92.67    97    87
Wales-S2       3   80.00    97    56
Wales-S3       3   82.33    97    56
Wales-S4       3   79.00    94    56
Wales-S5       3   92.67    97    87

Subject        Count     Avg  High   Low
PROGRAMMING        3   80.00    97    56
MATHEMATICS        3   79.00    94    56
ELECTRONICS        3   95.00    97    94
PHYSICS            3   76.67    87    56
ENGLISH            3   96.00    97    94


In [7]:
print("Students ranked by average:")
for rank, (st, s) in enumerate(sorted(student_summary.items(), key=lambda x: x[1]["avg"], reverse=True), 1):
    print(f"  {rank}. {st:<10}{s['avg']:>8.2f}")

print("\nSubjects ranked by average:")
for rank, (sj, s) in enumerate(sorted(subject_summary.items(), key=lambda x: x[1]["avg"], reverse=True), 1):
    print(f"  {rank}. {sj:<14}{s['avg']:>8.2f}")

print("\nRecords ordered by score (highest first):")
for r in sorted(valid, key=lambda r: r["score"], reverse=True):
    print(f"  {r['student']:<10}{r['subject']:<14}{r['score']:>6}  {r['category']}")

Students ranked by average:
  1. Wales-S1     92.67
  2. Wales-S5     92.67
  3. Wales-S3     82.33
  4. Wales-S2     80.00
  5. Wales-S4     79.00

Subjects ranked by average:
  1. ENGLISH          96.00
  2. ELECTRONICS      95.00
  3. PROGRAMMING      80.00
  4. MATHEMATICS      79.00
  5. PHYSICS          76.67

Records ordered by score (highest first):
  Wales-S1  PROGRAMMING       97  Excellent
  Wales-S2  ELECTRONICS       97  Excellent
  Wales-S3  ENGLISH           97  Excellent
  Wales-S5  ENGLISH           97  Excellent
  Wales-S1  ELECTRONICS       94  Excellent
  Wales-S3  ELECTRONICS       94  Excellent
  Wales-S4  ENGLISH           94  Excellent
  Wales-S5  MATHEMATICS       94  Excellent
  Wales-S1  MATHEMATICS       87  Good
  Wales-S2  PHYSICS           87  Good
  Wales-S4  PHYSICS           87  Good
  Wales-S5  PROGRAMMING       87  Good
  Wales-S2  MATHEMATICS       56  Needs Improvement
  Wales-S3  PHYSICS           56  Needs Improvement
  Wales-S4  PROGRAMMING     

In [8]:
top_student = max(student_summary, key=lambda s: student_summary[s]["avg"])
low_student = min(student_summary, key=lambda s: student_summary[s]["avg"])
top_subject = max(subject_summary, key=lambda s: subject_summary[s]["avg"])
low_subject = min(subject_summary, key=lambda s: subject_summary[s]["avg"])

print("=== FINAL STUDENT PERFORMANCE REPORT ===")
print("Total records generated :", len(records))
print("Valid, unique records   :", len(valid))
print("Repeated records        :", len(repeated))
print("Incomplete records      :", len(incomplete))
print("Highest-average student :", top_student, "->", round(student_summary[top_student]["avg"], 2))
print("Lowest-average student  :", low_student, "->", round(student_summary[low_student]["avg"], 2))
print("Highest-average subject :", top_subject, "->", round(subject_summary[top_subject]["avg"], 2))
print("Lowest-average subject  :", low_subject, "->", round(subject_summary[low_subject]["avg"], 2))

=== FINAL STUDENT PERFORMANCE REPORT ===
Total records generated : 17
Valid, unique records   : 15
Repeated records        : 1
Incomplete records      : 1
Highest-average student : Wales-S1 -> 92.67
Lowest-average student  : Wales-S4 -> 79.0
Highest-average subject : ENGLISH -> 96.0
Lowest-average subject  : PHYSICS -> 76.67
